# Score custom structures against PLINDER

`score_custom_cif_files()` annotates one or more custom mmCIF structures, searches their protein chains against PLINDER, and calculates the applicable protein, pocket, ligand, and protein-interface similarities.

## Input requirements

For an already assembled model, use `structure_mode="as_is"`. The `_atom_site` category must contain `group_PDB`, `type_symbol`, Cartesian coordinates, and the four label identifiers for atom, component, chain, and residue. Model number and insertion code may be omitted for a single-model structure. Deposition authors, citations, experimental details, and validation categories are not required.

Use `structure_mode="pdb"` only for a deposited-style asymmetric unit with `_entry.id`, assembly-generation records, and operation matrices. Missing required fields produce an error naming the absent category or column.

Unknown ligands such as `LIG` must have bond orders in `_chem_comp_bond`, or be assigned either a SMILES string or a CCD template. PLINDER writes bond-aware query SDFs and never modifies the input mmCIF.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PLINDER_RELEASE", "2026-07")
os.environ.setdefault("PLINDER_RELEASE_NUMBER", "1")

import pandas as pd

from plinder.core import PlinderSystem
from plinder.core.scores import (
    CustomProteinSearchConfig,
    score_custom_cif_files,
    score_custom_sequence_file,
)

release_path = os.environ.get("PLINDER_DATA_DIR")
release_path = Path(release_path) if release_path else None
input_cif = (
    Path(os.environ["PLINDER_CUSTOM_CIF"])
    if "PLINDER_CUSTOM_CIF" in os.environ
    else PlinderSystem(system_id="4agi__1__1.C__1.W").receptor_cif
)
work_dir = Path(
    os.environ.get("PLINDER_CUSTOM_WORK_DIR", "custom_scoring_example")
)

## Run a small receptor-only example

The example rebuilds a PLINDER receptor mmCIF when `PLINDER_CUSTOM_CIF` is not set. `plinder_entry_ids` limits the comparison to receptor and interface proteins from the selected entries; omit it to search all PLINDER proteins.

With `include_ligands=None` and `include_interfaces=None`, the workflow writes those score tables only when the custom input contains the corresponding feature.

In [ ]:
%%capture
result = score_custom_cif_files(
    [input_cif],
    work_dir=work_dir,
    data_dir=release_path,
    include_ligands=None,
    include_interfaces=None,
    backends=("mmseqs",),
    plinder_entry_ids=["4agi"],
    search_config=CustomProteinSearchConfig(max_seqs=25),
    threads=2,
    store_aligned_pocket_residues=True,
)

In [ ]:
protein_scores = pd.read_parquet(result.protein_scores)
residue_pairs = pd.read_parquet(result.aligned_pocket_residues)
{
    "protein_score_rows": len(protein_scores),
    "aligned_pocket_residue_rows": len(residue_pairs),
    "ligand_scores_written": result.ligand_scores is not None,
    "interface_scores_written": result.interface_scores is not None,
}

Protein scoring is oriented from each PLINDER receptor and ligand pocket to a custom protein chain. `pocket_fident` is therefore the percentage of PLINDER pocket residues that align to identical residues in the custom chain.

In [ ]:
pocket_scores = protein_scores.loc[
    protein_scores["metric"].astype(str).eq("pocket_fident"),
    [
        "query_system",
        "query_ligand_id",
        "target_system",
        "protein_mapping",
        "source",
        "similarity",
    ],
].head()
pocket_scores

When aligned pocket residues are requested, each row records a PLINDER pocket residue and the custom residue selected by the chain mapping and search source used for the reported score.

In [ ]:
residue_pairs[[
    "plinder_system_id",
    "plinder_chain_instance",
    "plinder_residue_number",
    "custom_structure_id",
    "custom_chain_asym_id",
    "custom_residue_number",
    "residue_identical",
]].head()

## Ligands with missing chemistry

Provide one chemistry source per custom component. The SMILES route maps parsed atoms positionally to the component atoms, while a CCD template uses CCD atom names and bonds. Score files with different meanings for `LIG` in separate calls. A proper ligand enables ligand-level pocket, interaction, Tanimoto, and—unless disabled—shape/SuCOS scoring.

In [ ]:
def score_ligand_model(
    cif_path,
    *,
    ligand_smiles_dict=None,
    ligand_ccd_code_dict=None,
):
    return score_custom_cif_files(
        [Path(cif_path)],
        work_dir=Path(cif_path).with_suffix("") / "plinder_scores",
        ligand_smiles_dict=ligand_smiles_dict,
        ligand_ccd_code_dict=ligand_ccd_code_dict,
    )


# Examples:
# score_ligand_model("model.cif", ligand_smiles_dict={"LIG": "CC(=O)N"})
# score_ligand_model("model.cif", ligand_ccd_code_dict={"LIG": "ATP"})

## Compare protein sequences

Protein sequences can be searched without coordinates. This MMseqs route writes protein scores, ligand-pocket links, the best link per input sequence, and optionally the aligned pocket residues.

In [ ]:
def score_sequences(fasta_path):
    return score_custom_sequence_file(
        Path(fasta_path),
        work_dir=Path(fasta_path).with_suffix("") / "plinder_scores",
        threads=8,
        store_aligned_pocket_residues=True,
    )

For several coordinate files, pass a list such as `sorted(Path("custom_cifs").glob("*.cif"))` to `score_custom_cif_files()`. Use `structure_mode="pdb"` with `assembly_ids=["1"]` for a deposited asymmetric unit; the default `as_is` treats model 1 as the complete supplied assembly.